# Class 5 (workshop version): Can a Language Model Do Our Jobs?

In **Class 1** you trained a classifier. In **Class 2** you fitted a line. Both needed
labelled data and a training step.

Today we ask a language model to do the same three jobs — **with no training at all**, and
then **with a handful of examples pasted into the question** — and we measure it.

| Part | Job | Data |
|---|---|---|
| 1 | Sentiment of customer reviews | 8 reviews (+ your own) |
| 2 | Which patients have the disease | Class 1's screening study |
| 3 | What is this house worth | Class 2's ten houses |
| 4 | What it cost | the bill |

**Two words you will use all day:**

- **Zero-shot** — you ask. No examples.
- **Few-shot** — you paste a few solved examples into the question first.

> ▶︎ Run cells with **Shift + Enter**, top to bottom. Cells marked ✏️ are **yours to edit**.
> There are twelve of them.

In [ ]:
# If running on Google Colab, clone the repo (if needed),
# move into the repo directory, and ensure it’s on the Python path.

import sys, os

def in_colab():
    try: import google.colab; return True
    except: return False

if in_colab():
    repo = "Hands-On-Notebooks"
    if os.path.basename(os.getcwd()) != repo:
        if not os.path.exists(repo):
            !git clone https://github.com/BridgingAISocietySummerSchools/{repo}
        %cd {repo}
    if '.' not in sys.path:
        sys.path.append('.')

In [ ]:
# Setup — run once.

import numpy as np

from plotting_utils.llm_simple import (
    REVIEWS, HOUSE_SIZES, HOUSE_PRICES,
    to_label, to_number, to_price_in_thousands,
    show_label_results, plot_accuracy_bars, create_classifier_playground,
)
from plotting_utils.llm_benchmark import (
    run_in_parallel, load_screening_benchmark, patient_to_text, labelled_examples,
    score_classification, show_scoreboard, plot_metric_comparison,
    score_regression, plot_price_comparison, plot_mae_bars, cost_projection,
)

print("✅ Ready.")

### 🔑 The key

Your instructor will give you one. It goes in the **hidden box** below — never in a cell.
(On Colab you can instead add `OPENROUTER_API_KEY` under 🔑 **Secrets**, with *Notebook
access* switched on.)

In [ ]:
import os, getpass
from llm_client import ask_llm, llm_available, describe_setup, print_usage, USAGE

if not llm_available():
    try:
        key = getpass.getpass("Paste the workshop key (hidden): ").strip()
    except Exception:
        key = ""
    if key:
        os.environ["OPENROUTER_API_KEY"] = key

# A small, fast model: under a second per answer, and cheap. Part 4 explains why.
os.environ["OPENROUTER_MODEL"] = "anthropic/claude-haiku-4.5"

LLM_READY = describe_setup()

In [ ]:
# ⏱️ Sizes. Small on purpose, so every cell finishes while you are still looking at it.

N_PATIENTS = 20      # patients we send to the model in Part 2
N_EXAMPLES = 15      # solved patients pasted into the prompt in Part 2
EX_PATIENTS = 10     # even smaller, for the ✏️ exercises

---
# Part 1 · Sentiment ⏱️ 30 min

The job: read a customer review, say **positive** or **negative**.

Class 1's way: collect thousands of labelled reviews, train, evaluate.
Today's way: ask.

In [ ]:
def ask(question):
    """Text in, text out. This is the whole interface."""
    return ask_llm(question, max_tokens=400)


print(ask("In two sentences, what is machine learning?"))

### ✏️ Exercise 1 — Ask it something you can check (3 min)

Replace the question with one **from your own field where you already know the answer**.
Then judge the answer. Was it right? Was it *confidently* wrong?

In [ ]:
# ✏️ EXERCISE 1 — change the text in the quotes.

print(ask("Explain what a p-value is to someone who has never studied statistics."))

### 🏷️ A classifier in six lines

Two things are doing the work:

- **"Answer with the category name only."** — otherwise you get a paragraph, and a paragraph
  is not a label.
- **`to_label(...)`** — turns *"Clearly positive."* back into `positive`.

In [ ]:
def classify(text, labels):
    """Sort `text` into exactly one of `labels`. No training, no examples."""
    prompt = (f"Classify the text into exactly one of these categories: {', '.join(labels)}.\n"
              f"Answer with the category name only.\n\n"
              f"Text: {text}")
    return to_label(ask_llm(prompt, max_tokens=20), labels)


print(classify("The battery lasts all day and the screen is gorgeous.", ["positive", "negative"]))

### ✏️ Exercise 2 — Predict, *then* run (5 min)

Write three reviews you think are **hard**. Sarcasm and faint praise are your best weapons.

**Before you run the cell:** write down what you expect for each one. Then run it. The
interesting reviews are the ones where you and the model disagree.

In [ ]:
# ✏️ EXERCISE 2 — edit these three lines. Predict first, then run.

my_tricky = [
    "Well, it arrived. Eventually.",
    "I have owned worse toasters.",
    "The colour is nice.",
]

for text in my_tricky:
    print(f"{classify(text, ['positive', 'negative']):>8}  |  {text}")

### 📏 Measuring it properly

Eight reviews with labels **we agreed in advance**. The model never sees the labels — they
exist only so we can grade it, exactly like Class 1's test set.

> 📌 We still need labelled data. Not to **train** the model — to **trust** it.

And it does not get to compete against nothing. Two baselines:

- **"Always positive"** — no reading required.
- **Keyword counting** — nice words vs nasty words. No AI, free, instant.

In [ ]:
for r in REVIEWS:
    print(f"{r['label']:>8}  |  {r['text']}")

In [ ]:
GOOD = ["great", "gorgeous", "excellent", "flawlessly", "best", "worth", "nice", "wonderful"]
BAD = ["broken", "cheap", "stopped", "crashes", "slow", "never"]

def keyword_rule(text):
    """No AI at all: count nice words vs nasty words."""
    t = text.lower()
    return "positive" if sum(w in t for w in GOOD) > sum(w in t for w in BAD) else "negative"


truth = [r["label"] for r in REVIEWS]
scores = {
    "Always 'positive'": sum(t == "positive" for t in truth) / len(truth),
    "Keyword counting": sum(keyword_rule(r["text"]) == t for r, t in zip(REVIEWS, truth)) / len(truth),
}

if LLM_READY:
    guesses = run_in_parallel(lambda r: classify(r["text"], ["positive", "negative"]), REVIEWS)
    scores["Language model"] = show_label_results([r["text"] for r in REVIEWS], truth, guesses)

plot_accuracy_bars(scores, title="Sentiment on the same 8 reviews")

### 💬 Two things to notice

1. **The last three reviews are the whole test.** Sarcasm, *good-then-bad*, *bad-then-good*.
   Keyword counting cannot ever get those; the model reads them the way you do.
2. **Our labels are opinions.** We *decided* "excellent sound, crashes daily" is negative. When
   the model disagrees with a label, ask first whether the label was right.

If the model scored 8/8, **the test is too easy to learn anything more from it.** So build a
harder one — that is the next exercise, and it is the single most useful habit here.

### ✏️ Exercise 3 — Build a harder test set (7 min)

Write **four reviews with the labels you believe are correct**. Aim for cases you had to think
about. Then grade the model on your set.

The four already in the cell are **faint praise**: every word is positive, the verdict is not.
Replace them with your own — mixed praise and complaint, sarcasm, a review in your second
language, one that is genuinely neutral, a five-word review.

> ⚠️ Two of the four are arguable. Is *"Exactly what I expected."* really negative? Hold on to
> that when the model "gets it wrong".

In [ ]:
# ✏️ EXERCISE 3 — write four reviews and YOUR label for each.

MY_REVIEWS = [
    {"text": "Exactly what I expected.",                                       "label": "negative"},
    {"text": "Five stars for the packaging.",                                  "label": "negative"},
    {"text": "Not the prettiest, but three years in and it has never failed.", "label": "positive"},
    {"text": "Works as advertised, which these days is a compliment.",         "label": "positive"},
]

# The three hard originals plus yours -- our benchmark from here on.
HARD = REVIEWS[5:] + MY_REVIEWS

if LLM_READY:
    hard_truth = [r["label"] for r in HARD]
    hard_guesses = run_in_parallel(lambda r: classify(r["text"], ["positive", "negative"]), HARD)
    zero_shot_hard = show_label_results([r["text"] for r in HARD], hard_truth, hard_guesses)

### ✏️ Exercise 4 — Fix it with the prompt (5 min)

You cannot retrain this model. You **can** rewrite the question. Add **one line** to the prompt
below and re-grade on the same hard set.

Things people try, roughly in order of how well they work:

- `"Look for faint praise and implied criticism: would the writer recommend it?"`
- `"Sarcasm is common. Judge the writer's overall verdict, not individual words."`
- `"If a review mentions both good and bad, the deciding factor is whether they would buy again."`

In [ ]:
# ✏️ EXERCISE 4 — add or change ONE line where marked.

def my_classify(text, labels):
    prompt = (f"Classify the text into exactly one of these categories: {', '.join(labels)}.\n"
              # ✏️ ADD YOUR LINE HERE -------------------------------------------------
              ""
              # -----------------------------------------------------------------------
              f"Answer with the category name only.\n\n"
              f"Text: {text}")
    return to_label(ask_llm(prompt, max_tokens=20), labels)


if LLM_READY:
    my_guesses = run_in_parallel(lambda r: my_classify(r["text"], ["positive", "negative"]), HARD)
    my_prompt_hard = show_label_results([r["text"] for r in HARD], hard_truth, my_guesses,
                                        note="— with my extra instruction.")

### 🎓 Few-shot: show it, don't tell it

Instead of *describing* what you want, **paste in solved examples**. That is called
**few-shot** (or *in-context learning*), and it is the second lever you have.

Note what it is not: this is **not training**. The examples are re-sent with every single
question, and the model forgets them the moment it answers.

In [ ]:
# ✏️ EXERCISE 5 — change these four examples. Try making them all sarcastic, or all mixed.

SENTIMENT_EXAMPLES = """\
"Took ages to arrive but I would buy it again." -> positive
"Looks lovely. Broke on day three." -> negative
"Sure, it works, if you enjoy a fight." -> negative
"Ten out of ten for the box it came in." -> negative"""


def classify_with_examples(text, labels):
    prompt = (f"Here are some reviews that have already been labelled:\n\n"
              f"{SENTIMENT_EXAMPLES}\n\n"
              f"Now classify this text as exactly one of: {', '.join(labels)}.\n"
              f"Answer with the category name only.\n\n"
              f"Text: {text}")
    return to_label(ask_llm(prompt, max_tokens=20), labels)


if LLM_READY:
    few_guesses = run_in_parallel(
        lambda r: classify_with_examples(r["text"], ["positive", "negative"]), HARD)
    few_shot_hard = show_label_results([r["text"] for r in HARD], hard_truth, few_guesses,
                                      note="— with 4 examples in the prompt.")

    plot_accuracy_bars({
        "Zero-shot": zero_shot_hard,
        "My prompt": my_prompt_hard,
        "Few-shot (4 examples)": few_shot_hard,
    }, title=f"On the same {len(HARD)} hard reviews")

### 💬 Which lever won?

There is no rule here, and that is the point. On a given task, sometimes a **better
instruction** wins, sometimes **examples** win, sometimes neither moves at all.

On seven reviews, **one review is 14 percentage points**. Unless one approach is clearly
ahead, you have not measured a difference — you have measured noise. Nobody can tell these
apart on a test set this small, and that includes the people selling you models.

### ✏️ Exercise 6 — The thing a trained model cannot do (5 min)

Class 1's classifier answers exactly one question. To make it sort support emails instead you
would need a new labelled data set and a new training run: **days**.

Here you change a sentence. Type any text and any categories, press **Run**.

Try: `urgent, normal` · `billing, technical, other` · `happy, angry, confused` ·
or categories from your own work.

In [ ]:
create_classifier_playground(classify)

---
# Part 2 · The Patients from Class 1 ⏱️ 30 min

Same job as Class 1: **does this patient have the disease?** Same data, same held-out
patients, same trained model.

Three competitors:

| | Sees the training data? |
|---|---|
| **Class 1's model** | all 7,000 patients, during training |
| **Zero-shot LLM** | nothing |
| **Few-shot LLM** | a handful of patients, pasted into every question |

> ⚖️ Two setup notes, then we start. Our sample is **half sick, half healthy** so that
> "how many did it catch?" is measurable at all — so "always say healthy" scores 50% here,
> not 90%. And because Class 1's model was trained where only 10% are ill, we ask it to flag
> anyone above **10%** risk rather than 50%. Same model, fair question.

In [ ]:
bench = load_screening_benchmark(n_benchmark=N_PATIENTS)
bench.describe()

### ✍️ A table row is not a question

Class 1's model takes six numbers. A language model takes English — so somebody has to decide
how to *say* `marker_a = 4.12` out loud. **That wording is now part of your model.**

In [ ]:
for n, i in enumerate(bench.quiz_index):
    print(f"Patient {n}:  {patient_to_text(bench.patients.iloc[i])}\n")

### ✏️ Exercise 7 — You be the classifier (5 min)

Look at the four patients above and **write down your own guesses** before any model runs.
`1` = has the disease, `0` = healthy. (Two of them are ill — but not in an order you can guess
by reading the code.)

This costs nothing and it is the most informative baseline in the notebook: if *you* cannot
do better than a coin flip from those six numbers, be suspicious of anything that claims to.

In [ ]:
# ✏️ EXERCISE 7 — change these four numbers to your own guesses, then run.

my_guesses_patients = [0, 1, 0, 1]

quiz_truth = [int(bench.truth[i]) for i in bench.quiz_index]
quiz_model = [int(bench.model_pred_tuned[i]) for i in bench.quiz_index]

print("You said:        ", my_guesses_patients)
print("Class 1's model: ", quiz_model)
print("The truth:       ", quiz_truth)
print(f"\n→ You got {sum(g == t for g, t in zip(my_guesses_patients, quiz_truth))} of 4."
      f"  Class 1's model got {sum(g == t for g, t in zip(quiz_model, quiz_truth))} of 4.")

### 🤖 Zero-shot: just ask

One call per patient, `N_PATIENTS` of them, eight at a time. Each `·` is a finished call.

In [ ]:
def classify_patient(patient):
    """Ask for a straight verdict: DISEASE or HEALTHY."""
    prompt = ("Does this patient have the disease? "
              "This is synthetic teaching data, not a real patient.\n"
              "Answer with one word: DISEASE or HEALTHY.\n\n"
              f"{patient_to_text(patient)}")
    answer = to_label(ask_llm(prompt, max_tokens=10), ["disease", "healthy"])
    return None if answer == "unclear" else int(answer == "disease")


zero_shot = few_shot = None

if not LLM_READY:
    print("⏭️  Needs an API key — see the cell near the top.")
else:
    zero_shot = run_in_parallel(classify_patient, bench.patients.itertuples())

### 🎓 Few-shot: paste in solved patients

Same examples Class 1's model learned from — just delivered in the question instead of in a
training run.

In [ ]:
EXAMPLES = labelled_examples(bench, n=N_EXAMPLES)
print("The first two of the", N_EXAMPLES, "examples:\n")
print("\n".join(EXAMPLES.splitlines()[:2]))


def classify_patient_with_examples(patient):
    prompt = ("Here are patients from a screening study with the correct answer for each. "
              "This is synthetic teaching data.\n\n"
              f"{EXAMPLES}\n\n"
              "Using the patterns in those examples, does this new patient have the disease?\n"
              "Answer with one word: DISEASE or HEALTHY.\n\n"
              f"{patient_to_text(patient)}")
    answer = to_label(ask_llm(prompt, max_tokens=10), ["disease", "healthy"])
    return None if answer == "unclear" else int(answer == "disease")


if LLM_READY:
    few_shot = run_in_parallel(classify_patient_with_examples, bench.patients.itertuples())

### 🏁 The scoreboard

Same patients, the metrics from Class 1. **Recall** is the one that matters for a screening
tool: *of the people who really had the disease, how many did we catch?*

In [ ]:
results = {
    "Always say 'healthy'": score_classification(bench.truth, bench.baseline_pred),
    "Class 1's trained model": score_classification(bench.truth, bench.model_pred_tuned),
}
if LLM_READY:
    results["LLM, zero-shot"] = score_classification(bench.truth, zero_shot)
    results[f"LLM, {N_EXAMPLES} examples"] = score_classification(bench.truth, few_shot)

show_scoreboard(results, title=f"Screening, on the same {N_PATIENTS} held-out patients")
plot_metric_comparison(results, metrics=("accuracy", "precision", "recall"))

### 💬 Why the trained model wins here

Look at the two blood markers. **They are invented.** Nowhere on the internet does it say what
`marker_a = 4.12` means — and in this data set the marker is where the signal is.

- The language model knows what any doctor knows about **age, BMI, smoking, family history**.
  That gets it off the floor, and it is real knowledge you did not have to pay for.
- It cannot know **marker_a**, so it quietly leans on the columns it recognises.
- Class 1's model learned the marker from 7,000 examples in three milliseconds.

> 📌 Your company's tables are full of `marker_a`: `customer_score_v3`, `region_code_7`.
> **Whoever has the labelled history wins on data like this.**

And check the few-shot row before you assume examples help. On six columns of numbers they
often make things **worse**: fifteen rows is not enough to see the pattern, so the model
starts flagging almost everyone. Recall goes up, precision collapses. That is the opposite of
what happens in Part 3 — which is why you measure instead of assuming.

Now check the confidence interval column. On 20 patients it spans about ±20 points — so be
careful about which of these differences you actually believe.

### ✏️ Exercise 8 — Give it knowledge it could not have (7 min)

Add **one line** to the prompt that tells the model something about this study. Then re-run on
a small sample and compare.

Ideas:

- `"Blood marker A is an inflammation marker; above 4.0 is considered elevated."`
- `"About 10% of patients in this study have the disease."`
- `"The marker matters most in patients over 60."`  ← this one is actually true here

**The question to answer:** if inventing a meaning for `marker_a` improves the score, what
have you learned — about the model, or about yourself?

In [ ]:
# ✏️ EXERCISE 8 — add your line where marked, then run.

def my_classify_patient(patient):
    prompt = ("Does this patient have the disease? This is synthetic teaching data.\n"
              # ✏️ ADD YOUR LINE HERE -------------------------------------------------
              ""
              # -----------------------------------------------------------------------
              "Answer with one word: DISEASE or HEALTHY.\n\n"
              f"{patient_to_text(patient)}")
    answer = to_label(ask_llm(prompt, max_tokens=10), ["disease", "healthy"])
    return None if answer == "unclear" else int(answer == "disease")


if LLM_READY:
    sample = bench.patients.head(EX_PATIENTS)
    mine = run_in_parallel(my_classify_patient, sample.itertuples())

    show_scoreboard({
        "Zero-shot (original)": score_classification(bench.truth[:EX_PATIENTS],
                                                    zero_shot[:EX_PATIENTS]),
        "My prompt": score_classification(bench.truth[:EX_PATIENTS], mine),
        "Class 1's model": score_classification(bench.truth[:EX_PATIENTS],
                                                bench.model_pred_tuned[:EX_PATIENTS]),
    }, title=f"On the first {EX_PATIENTS} patients")

### ✏️ Exercise 9 — Do more examples help? (5 min)

Few-shot has an obvious knob: **how many examples**. Change the number below and see.

Predict first: 3 examples versus 15 — how much difference do you expect, and in which
direction? Then check whether what you got is bigger than the confidence interval.

> 💡 More is not always better. Fifteen rows of six numbers can leave the model *less* sure
> than three, not more.

In [ ]:
# ✏️ EXERCISE 9 — change this number, then run.

HOW_MANY = 3

if LLM_READY:
    EXAMPLES = labelled_examples(bench, n=HOW_MANY)      # rebuilds the block used above
    sample = bench.patients.head(EX_PATIENTS)
    tried = run_in_parallel(classify_patient_with_examples, sample.itertuples())

    show_scoreboard({
        f"{HOW_MANY} examples": score_classification(bench.truth[:EX_PATIENTS], tried),
        f"{N_EXAMPLES} examples": score_classification(bench.truth[:EX_PATIENTS],
                                                      few_shot[:EX_PATIENTS]),
        "Class 1's model": score_classification(bench.truth[:EX_PATIENTS],
                                                bench.model_pred_tuned[:EX_PATIENTS]),
    }, title=f"How many examples? (first {EX_PATIENTS} patients)")

EXAMPLES = labelled_examples(bench, n=N_EXAMPLES)        # put it back, so later cells match

---
# Part 3 · The Houses from Class 2 ⏱️ 20 min

Classification asked *which category?* Regression asks **how much?**

Class 2 had ten house sales. We split them **five and five**:

- **5 houses we are allowed to learn from** — Class 2's line is fitted on these, and they are
  the ones we paste into the few-shot prompt.
- **5 houses nobody sees** — everyone predicts these.

Same information, three ways of using it. That is a fair fight.

In [ ]:
from sklearn.linear_model import LinearRegression

sizes, prices = np.array(HOUSE_SIZES), np.array(HOUSE_PRICES)
shown, hidden = np.array([0, 2, 4, 6, 8]), np.array([1, 3, 5, 7, 9])

print("🏠 The 5 houses everyone gets to see:")
for s, p in zip(sizes[shown], prices[shown]):
    print(f"   {s:,} sq ft sold for ${p}k")

print("\n❓ The 5 houses we have to predict:")
for s in sizes[hidden]:
    print(f"   {s:,} sq ft → ?")

line = LinearRegression().fit(sizes[shown].reshape(-1, 1), prices[shown])
line_guesses = line.predict(sizes[hidden].reshape(-1, 1))
print(f"\n📏 Class 2's line, fitted on those 5: ${line.coef_[0] * 1000:.0f} per square foot.")

### ✏️ Exercise 10 — Beat the line (5 min)

Look at the five known sales and **write down your own guess** for each hidden house, in
thousands. Then run — you are scored alongside everyone else.

In [ ]:
# ✏️ EXERCISE 10 — replace these five numbers with your own guesses.

my_price_guesses = [180, 250, 300, 380, 420]

print("Size      you    real")
for s, g, real in zip(sizes[hidden], my_price_guesses, prices[hidden]):
    print(f"{s:>5,}   {g:>4}   {real:>4}    (off by ${abs(g - real) * 1000:,})")

print(f"\nYour average miss: ${score_regression(prices[hidden], my_price_guesses)['mae'] * 1000:,.0f}")

### 🤖 Zero-shot: no data at all

The model has never seen our neighbourhood. It has to silently pick a country, a decade and a
market before it can answer.

In [ ]:
def price_zero_shot(size):
    prompt = (f"A house is {size} square feet. Estimate its price in thousands of US dollars.\n"
              f"Answer with just a number, no words, no dollar sign. Example: 250")
    return to_price_in_thousands(ask_llm(prompt, max_tokens=20))


llm_zero = llm_few = None
if LLM_READY:
    llm_zero = run_in_parallel(price_zero_shot, sizes[hidden])
    print("\nIts guesses:", llm_zero)

### 🎓 Few-shot: the same five sales the line got

In [ ]:
SALES = "\n".join(f"{s:,} sq ft sold for ${p}k" for s, p in zip(sizes[shown], prices[shown]))


def price_few_shot(size):
    prompt = (f"Here are recent house sales in one neighbourhood:\n{SALES}\n\n"
              f"Estimate the price of a {size:,} sq ft house in the same neighbourhood, "
              f"in thousands of dollars.\n"
              f"Answer with just a number, no words, no dollar sign. Example: 250")
    return to_price_in_thousands(ask_llm(prompt, max_tokens=20))


if LLM_READY:
    llm_few = run_in_parallel(price_few_shot, sizes[hidden])

In [ ]:
series = {"Class 2's line": line_guesses, "Your guesses": my_price_guesses}
if LLM_READY:
    series["LLM, zero-shot"] = llm_zero
    series["LLM, 5 examples"] = llm_few

plot_price_comparison(sizes[hidden], prices[hidden], series,
                      title="The 5 hidden houses")

errors = {name: score_regression(prices[hidden], values) for name, values in series.items()}
plot_mae_bars(errors)
for name, e in errors.items():
    print(f"{name:>18}:  average miss ${e['mae'] * 1000:,.0f}")

### 💬 What the zero-shot error is really measuring

Not house prices. **Which market it guessed.** If it happened to assume somewhere like ours it
looks brilliant; if it assumed San Francisco it looks absurd — and neither outcome tells you
anything about whether it can do regression.

Give it the five sales and it usually lands close to the line. Read five numbers, infer the
pattern, no fitting step — that is real, and it is why "I need a rough model by lunchtime" is
now a solvable problem.

But the line still hands you **`$167 per square foot`**: a sentence you can argue with, check,
and defend. Ask the model why it said 350 and you get a fluent paragraph that may have nothing
to do with how it got there.

### ✏️ Exercise 11 — Push them off the edge of the data (5 min)

Class 2 warned about **extrapolation**: our houses run 800–2,600 sq ft. The line has no idea
what happens outside that, and it will not hesitate to tell you anyway.

Try `5000`. Then `200`. Then `50000`. Who stays sensible, and who is confidently absurd?

In [ ]:
# ✏️ EXERCISE 11 — change the size, then run.

BIG = 5000

full_line = LinearRegression().fit(sizes.reshape(-1, 1), prices)
print(f"Class 2's line:  ${full_line.predict([[BIG]])[0]:,.0f}k")

if LLM_READY:
    print(f"LLM (5 examples): ${price_few_shot(BIG):,.0f}k")

print("\n🤔 Would you put either number in a report?")

---
# Part 4 · The Bill ⏱️ 10 min

Accuracy is the column everyone reports. **Cost and speed decide whether it ships.**

In [ ]:
import time

llm_seconds = float("nan")
if LLM_READY:
    start = time.time()
    ask_llm("Reply with the single word: ready.", max_tokens=5)
    llm_seconds = time.time() - start

start = time.time()
bench.model.predict(bench.X_test)
model_seconds = (time.time() - start) / len(bench.X_test)

print(f"⏱️  Language model:     {llm_seconds:.2f} seconds per prediction")
print(f"⏱️  Class 1's model:    {model_seconds * 1e6:.1f} microseconds per prediction\n")
print_usage()
print()
cost_projection(USAGE, predictions_per_day=10_000, seconds_per_call=llm_seconds)

### 💬 And that was already the cheap model

We ran everything on a small, fast one. A frontier model would multiply that bill by about
ten for the same work.

Two costs that surprise people:

- **The examples are charged every time.** Those example patients were re-sent with every
  single question — look at the input-token count above. Class 1's model absorbed 7,000
  patients once, and has charged nothing since.
- **Speed is a cost too.** Read the last line above: that is how long 10,000 predictions take
  if you run them one at a time.

---
# ✅ Wrap-Up

| | **Train a small model** | **Ask a language model** |
|---|---|---|
| Labelled data | thousands of rows | none, or a handful |
| Time to first version | days | minutes |
| Free text, no labels yet | ❌ impossible | ✅ **won Part 1 outright** |
| Table of numbers with history | ✅ **won Part 2** | ❌ |
| Cost per prediction | ~free | a fraction of a cent, **forever** |
| Speed | microseconds | ~1 second |
| Can you read the rule? | ✅ `$167 per sq ft` | ❌ |
| Change the task | retrain | rewrite one sentence |

### 🧭 Four ideas worth keeping

1. **Same model, best and worst tool in the room, twenty minutes apart.** "Is AI good at
   this?" has no answer. *"Good at what, measured how?"* does.
2. **Zero-shot brings world knowledge. It cannot bring knowledge of your columns.** `age` it
   knows; `marker_a` it cannot.
3. **Few-shot is cheap to try and unreliable to depend on.** It transformed the houses and did
   much less for the patients. You only find out by measuring.
4. **You still need labelled data — to *trust* it, not to *train* it.** Everything honest in
   this notebook came from held-out answers.

### ✏️ Exercise 12 — Your own task (10 min, no code)

Think of one real task from **your** work or studies. Fill this in, out loud or in the cell
below.

| Question | Your answer |
|---|---|
| What is the input, and what is the output? | |
| Is the input **free text/images**, or **numbers in a table**? | |
| Do labelled examples already exist? How many? | |
| How many predictions per day? | |
| What happens when it is **wrong**? | |
| Based on Parts 1–3: **train a model, or prompt one?** | |

> Then argue the opposite case for two minutes. If you cannot, you have not understood the
> trade-off yet.

In [ ]:
# ✏️ EXERCISE 12 — notes, if you want them somewhere.

my_task = """
Input:
Output:
Text or table?
Labelled examples available:
Predictions per day:
Cost of being wrong:
My decision:
"""
print(my_task)

### 📚 Next

- [`05_agentic_ai_benchmark.ipynb`](05_agentic_ai_benchmark.ipynb) — the same comparison done
  properly: risk scores, thresholds, ROC curves and confidence intervals.
- [`05_agentic_ai.ipynb`](05_agentic_ai.ipynb) — **RAG**: giving a model your own documents so
  it stops guessing. Plus tool-using agents and prompt injection.